### Stochastic Lagrangian Control and Proportional Bias Routing for Mixture-of-Experts

In this notebook we benchmark and contrast two auxiliary-loss-free routing methodologies:
- **Stochastic Augmented Lagrangian Control (SLC):** Casts token routing as a formal constrained optimization problem, employing dual variables (prices) and an augmented Lagrangian function to enforce strict expert capacity constraints via a robust two-timescale saddle-point update.
- **Proportional Bias Routing:** Formalizes the baseline control strategy popularized by DeepSeek-V3, utilizing mean-preserving proportional feedback integrators driven by hard token dispatch counts.

Read the paper [Stochastic Lagrangian Control and Proportional Bias Routing for Mixture-of-Experts](https://arxiv.org/abs/2305.12714) for more details.

In [1]:
import time
import torch
from routers import make_router

In [2]:
def run_benchmark(variant: str, steps: int = 200) -> None:
    latent_dim = 128
    num_experts = 16
    k = 2
    batch_size = 32
    seq_len = 64
    n_tokens = batch_size * seq_len

    kwargs = {"latent_dim": latent_dim, "num_experts": num_experts}
    if variant == "SLC":
        kwargs["warmup_steps"] = 10

    router = make_router(variant, **kwargs)
    optimizer = torch.optim.Adam(router.parameters(), lr=1e-3)

    print(f"\n--- Benchmarking Router Variant: {variant} ---")
    start_time = time.time()

    for step in range(steps):
        # Inject artificial skew into token distribution to test control recovery
        x = torch.randn(n_tokens, latent_dim)
        if step < steps // 2:
            x[:, :4] += 2.0  # Skew toward first few experts initially

        optimizer.zero_grad()
        gates, indices, expected_counts = router(x, k=k, return_aux=True)

        # Dummy primal loss favoring gate magnitudes
        loss = -(gates * router.affinities.gather(1, indices)).mean()
        loss.backward()
        optimizer.step()

        counts = torch.bincount(indices.view(-1), minlength=num_experts).float()
        router.update(
            counts=counts,
            expected_counts=expected_counts,
            n_tokens=n_tokens,
            top_k=k,
        )

        if (step + 1) % 50 == 0:
            m = router.metrics()
            print(f"Step [{step+1}/{steps}] | Primal: {m['primal']:.4f} | Dual: {m['dual']:.4f} | Gap: {m['abs_gap']:.4f}")

    elapsed = time.time() - start_time
    print(f"Completed {steps} steps in {elapsed:.4f} seconds.")

In [3]:
torch.manual_seed(42)

# Stochastic Augmented Lagrangian Control (SLC) Router.
run_benchmark("SLC", steps=150)

# DeepSeek Proportional Bias Router.
run_benchmark("DeepSeekBias", steps=150)


--- Benchmarking Router Variant: SLC ---
Step [50/150] | Primal: 0.3029 | Dual: 0.3036 | Gap: 0.0007
Step [100/150] | Primal: 0.3026 | Dual: 0.3025 | Gap: 0.0001
Step [150/150] | Primal: 0.3023 | Dual: 0.3024 | Gap: 0.0000
Completed 150 steps in 0.3850 seconds.

--- Benchmarking Router Variant: DeepSeekBias ---
Step [50/150] | Primal: 0.3399 | Dual: 0.3274 | Gap: 0.0124
Step [100/150] | Primal: 0.3392 | Dual: 0.3374 | Gap: 0.0018
Step [150/150] | Primal: 0.3386 | Dual: 0.3395 | Gap: 0.0010
Completed 150 steps in 0.4393 seconds.
